# Kiểm tra cấu hình Kaggle offline cho notebook 1 và 3

Mở **phiên Kaggle mới**, chọn đúng GPU dự định dùng và **Internet Off**.
Gắn các Dataset **models, packages và PDF** hiện tại rồi **Run All**; không cần chạy
setup notebook 1/3 trước. Nếu chưa gắn Input, vẫn lấy được báo cáo kernel.

Notebook kiểm tra Python/ABI, CUDA/VRAM, toolchain, thư viện OCR/restoration,
CogView4/GLM/PEFT, FlashAttention và các runtime riêng đã tồn tại.
**Không cài/build package, không tải hoặc nạp weights, không chạy OCR/restoration.**
Các phép thử GPU chỉ dùng tensor tổng hợp rất nhỏ. Mỗi phép có timeout; tổng ngân
sách mặc định 10 phút. JSON được lưu sau mỗi phép để giữ kết quả khi bị ngắt.

Metadata, file trong Input, import, tensor GPU và inference model là các mức kiểm
chứng khác nhau. Thiếu package trong kernel có thể được giải quyết bằng runtime
riêng; báo cáo này không tự kết luận cả pipeline đã chạy được.


In [ ]:
from pathlib import Path

ON_KAGGLE = Path('/kaggle/working').is_dir()
INPUT_ROOT = Path('/kaggle/input') if ON_KAGGLE else Path('kaggle_upload')
# Để trống để tự tìm qua manifest; nếu có nhiều Dataset cùng loại, điền đường dẫn.
PACKAGES_DIR = ''
MODELS_DIR = ''
DATA_DIR = ''
PREPARED_IMAGES = ''  # tùy chọn, thư mục có dataset.json; chỉ đọc descriptor

# Chỉ điền nếu runtime riêng ĐÃ tồn tại; notebook không tạo/cài runtime.
RUNTIME_PYTHONS = {
    # 'difftsr': '/path/to/verified/environment/bin/python',
    # 'unihir_restoration': '/path/to/verified/environment/bin/python',
    # 'tadisr_x4': '/path/to/overlay/bin/python',
}
RUN_PROBES = True
CHECK_OPTIONS = {
    'verify_asset_hashes': False,  # mặc định chỉ stat kích thước; không đọc hàng chục GB weights
    'check_existing_runtimes': True,
    'max_runtime_checks': 4,
    'max_scan_dirs': 500,
    'max_asset_records': 12000,
    'probe_timeout_cap': 180,
    'time_budget_seconds': 600,
}
OUTPUT_DIR = Path('/kaggle/working') if ON_KAGGLE else Path('work/kaggle_environment_check')


In [ ]:
"""Read-only environment inventory; no installs, downloads, weights or OCR."""
from __future__ import annotations

from datetime import datetime, timezone
import importlib.metadata as metadata
import importlib.util
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import sysconfig
import hashlib
import time
from pathlib import PurePosixPath


PACKAGES = (
    "torch", "torchvision", "transformers", "tokenizers", "accelerate", "safetensors",
    "huggingface-hub", "sentencepiece", "protobuf", "numpy", "scipy", "pillow", "opencv-python", "opencv-python-headless",
    "paddlepaddle", "paddlepaddle-gpu", "paddleocr", "paddlex", "mmcv", "mmcv-full",
    "mmcv-lite", "mmengine", "mmdet", "mmrotate", "timm", "einops", "flash-attn",
    "triton", "pyyaml", "pypdfium2", "pandas", "tqdm", "pip", "setuptools", "wheel",
    "packaging", "ninja", "cmake", "uv", "diffusers", "peft", "omegaconf",
    "pytorch-lightning", "lightning-utilities", "torchmetrics", "zhconv", "fonttools",
    "decord", "bitsandbytes", "sentence-transformers", "opencv-contrib-python",
)

PROBES = {
    "torch_cuda": ("torch", 90, """import torch, json
print('stage: torch imported', flush=True)
devices = [{'name': torch.cuda.get_device_name(i), 'VRAM_GiB': round(torch.cuda.get_device_properties(i).total_memory/1024**3,2), 'compute_capability': list(torch.cuda.get_device_capability(i))} for i in range(torch.cuda.device_count())]
result = {'torch': torch.__version__, 'built_cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'devices': devices,
          'compiled_arches': getattr(torch.cuda, 'get_arch_list', lambda: [])() if torch.cuda.is_available() else [],
          'module_file': torch.__file__, 'native_file': torch._C.__file__,
          'cxx11_abi': getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', None)}
if torch.cuda.is_available():
    x = torch.ones((2,2), device='cuda:0')
    result['gpu_matmul_sum'] = float((x @ x).sum().item())
print(json.dumps(result))
assert result['cuda_available'], 'CUDA unavailable; this is not a verified GPU runtime'
"""),
    "torchvision_ops": ("torchvision", 90, """import torch, torchvision, json
from torchvision.ops import nms
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[0.,0.,2.,2.],[0.,0.,2.,2.]],device=device)
keep = nms(boxes, torch.tensor([.9,.8],device=device), .5)
assert keep.numel() == 1
print(json.dumps({'torchvision': torchvision.__version__, 'nms_device': device}))
"""),
    "torch_numpy_bridge": ("torch", 45, """import torch, numpy as np, json
x = torch.from_numpy(np.zeros((2,2),dtype=np.float32))
assert x.shape == (2,2) and x.numpy().shape == (2,2)
print(json.dumps({'torch':torch.__version__,'numpy':np.__version__,'numpy_bridge':'OK'}))
"""),
    "numpy_cv2": ("cv2", 45, """import numpy as np, cv2, json
from PIL import Image
x = np.zeros((4,4,3),dtype=np.uint8)
assert cv2.cvtColor(x,cv2.COLOR_RGB2GRAY).shape == (4,4)
assert Image.fromarray(x).size == (4,4)
print(json.dumps({'numpy': np.__version__, 'opencv': cv2.__version__, 'array_image_bridge': 'OK'}))
"""),
    "transformers_hunyuan": ("transformers", 180, """import json
print('stage: transformers import', flush=True)
import transformers
print('stage: AutoProcessor', flush=True)
from transformers import AutoProcessor
print('stage: Hunyuan class', flush=True)
from transformers import HunYuanVLForConditionalGeneration
print(json.dumps({'transformers': transformers.__version__, 'hunyuan_api': 'OK'}))
"""),
    "paddle_cuda": ("paddle", 90, """import paddle, json
result = {'paddle': paddle.__version__, 'compiled_with_cuda': paddle.is_compiled_with_cuda()}
result['devices'] = paddle.device.cuda.device_count() if result['compiled_with_cuda'] else 0
if result['devices']:
    paddle.set_device('gpu:0')
    x = paddle.ones((2,2))
    result['gpu_matmul_sum'] = float(paddle.matmul(x,x).sum().item())
print(json.dumps(result))
"""),
    "paddleocr_apis": ("paddleocr", 120, """import paddleocr, json
from paddleocr import PaddleOCR, PaddleOCRVL
print(json.dumps({'paddleocr': paddleocr.__version__, 'PaddleOCR': 'OK', 'PaddleOCRVL': 'OK'}))
"""),
    "mmcv_compiled_ops": ("mmcv", 120, """import torch, mmcv, mmcv.ops, mmdet, mmrotate, json
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
boxes = torch.tensor([[5.,5.,2.,2.,0.]],device=device)
scores = torch.tensor([.9],device=device)
_, keep = mmcv.ops.nms_rotated(boxes,scores,.5)
assert keep.numel() == 1
print(json.dumps({'mmcv': mmcv.__version__, 'mmdet': mmdet.__version__, 'mmrotate': mmrotate.__version__, 'rotated_nms_device': device}))
"""),
    "pdf_renderer": ("pypdfium2", 45, """import pypdfium2 as pdfium, json
doc = pdfium.PdfDocument.new()
page = doc.new_page(32,32)
bitmap = page.render(scale=1)
image = bitmap.to_pil()
assert image.size == (32,32)
image.close(); bitmap.close(); page.close(); doc.close()
print(json.dumps({'blank_page_32x32': 'OK'}))
"""),
    "torch_attention": ("torch", 90, """import torch, json
from torch.nn.functional import scaled_dot_product_attention
assert torch.cuda.is_available(), 'CUDA required for attention probe'
tested = []
for dtype in [torch.float16] + ([torch.bfloat16] if torch.cuda.is_bf16_supported() else []):
    q = torch.randn(1, 2, 16, 64, device='cuda', dtype=dtype)
    y = scaled_dot_product_attention(q, q, q)
    torch.cuda.synchronize()
    assert y.shape == q.shape and bool(torch.isfinite(y).all())
    tested.append(str(dtype))
print(json.dumps({'tiny_sdpa_cuda': 'OK', 'tested_dtypes': tested, 'model_inference': False}))
"""),
    "diffusers_cogview4": ("diffusers", 180, """import json, diffusers
print('stage: CogView4 classes', flush=True)
from diffusers import CogView4Transformer2DModel, CogView4Pipeline
print(json.dumps({'diffusers':diffusers.__version__, 'module_file':diffusers.__file__,
 'CogView4Transformer2DModel':'OK', 'CogView4Pipeline':'OK', 'weights_loaded':False}))
"""),
    "transformers_glm": ("transformers", 180, """import json, transformers
from transformers import GlmModel
print(json.dumps({'transformers':transformers.__version__, 'GlmModel':'OK', 'weights_loaded':False}))
"""),
    "peft_api": ("peft", 120, """import peft, json
from peft import PeftModel
print(json.dumps({'peft':peft.__version__, 'module_file':peft.__file__, 'PeftModel':'OK'}))
"""),
    "restoration_dependencies": ("accelerate", 90, """import importlib, json
names = ['accelerate','safetensors','sentencepiece','huggingface_hub','PIL','numpy']
result = {}
for name in names:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps(result))
"""),
    "difftsr_dependencies": ("omegaconf", 120, """import importlib, json
result = {}
for name in ['torch','torchvision','pytorch_lightning','omegaconf','zhconv','einops','torchmetrics']:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps({'third_party_imports':result,'difftsr_model_code_tested':False}))
"""),
    "unihir_dependencies": ("accelerate", 120, """import importlib, json
result = {}
for name in ['torch','transformers','accelerate','fontTools','safetensors','einops']:
    module = importlib.import_module(name)
    result[name] = {'version':getattr(module,'__version__',None), 'file':getattr(module,'__file__',None)}
print(json.dumps({'third_party_imports':result,'unihir_model_code_tested':False}))
"""),
    "flash_attention": ("flash_attn", 90, """import torch, flash_attn, json
from flash_attn import flash_attn_func
assert torch.cuda.is_available(), 'CUDA required for FlashAttention probe'
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
q = torch.randn(1,16,2,64,device='cuda',dtype=dtype)
y = flash_attn_func(q,q,q)
torch.cuda.synchronize()
assert y.shape == q.shape and bool(torch.isfinite(y).all())
print(json.dumps({'flash_attn':flash_attn.__version__, 'dtype':str(dtype),
 'module_file':flash_attn.__file__, 'tiny_cuda_forward':'OK', 'model_inference':False}))
"""),
    "torch_build": ("torch", 90, """import torch, json, sysconfig, pathlib
from torch.utils.cpp_extension import CUDA_HOME
print(json.dumps({'CUDA_HOME':str(CUDA_HOME) if CUDA_HOME else None,
 'torch_build_config':torch.__config__.show(), 'Python_h_exists':pathlib.Path(sysconfig.get_path('include'),'Python.h').is_file(),
 'compiled_arches':torch.cuda.get_arch_list() if torch.cuda.is_available() else [], 'extension_compilation_performed':False}))
"""),
    "torch_restoration_ops": ("torch", 90, """import torch, json
from torch.nn import functional as F
assert torch.cuda.is_available(), 'CUDA required for restoration tensor ops'
x = torch.ones(1,3,16,16,device='cuda',dtype=torch.float32)
y = F.conv2d(x,torch.ones(12,3,3,3,device='cuda'),padding=1)
y = F.pixel_shuffle(y,2)
y = F.interpolate(y,scale_factor=2,mode='bicubic',align_corners=False)
torch.cuda.synchronize()
assert tuple(y.shape)==(1,3,64,64) and bool(torch.isfinite(y).all())
print(json.dumps({'conv2d_pixel_shuffle_bicubic_cuda':'OK','dtype':'float32','model_inference':False}))
"""),
}


PROBE_GUARD = '''import sys
sys.dont_write_bytecode = True
def _vtm_diagnostic_audit(event, args):
    if event in ('socket.connect', 'socket.getaddrinfo'):
        raise RuntimeError('Network disabled in environment diagnostics')
sys.addaudithook(_vtm_diagnostic_audit)
'''

DIAGNOSTIC_DEFAULTS = {'max_scan_dirs': 500, 'max_asset_records': 12000,
    'verify_asset_hashes': False, 'check_existing_runtimes': True,
    'max_runtime_checks': 4, 'probe_timeout_cap': 180, 'time_budget_seconds': 600}

RUNTIME_INVENTORY = '''import sys, sysconfig, platform, json, importlib.metadata as md, importlib.util
records, seen = [], set()
for dist in md.distributions():
    name=dist.metadata.get('Name',''); key=name.lower().replace('_','-').replace('.','-')
    records.append({'name':name,'version':dist.version,'active':key not in seen,
        'location':str(dist.locate_file('')),'requires_python':dist.metadata.get('Requires-Python'),
        'requires':dist.requires or []})
    seen.add(key)
modules={}
for name in ['torch','torchvision','transformers','numpy','cv2','paddle','paddleocr','mmcv',
             'diffusers','peft','accelerate','omegaconf','flash_attn','pypdfium2','fontTools']:
    try:
        spec=importlib.util.find_spec(name)
        modules[name]={'available':spec is not None,'origin':getattr(spec,'origin',None)}
    except Exception as exc: modules[name]={'available':False,'error':str(exc)}
try:
    from packaging.tags import sys_tags
    tags=[str(tag) for tag in sys_tags()]
except Exception: tags=[]
print(json.dumps({'python':sys.version,'executable':sys.executable,'prefix':sys.prefix,
    'ABI':sysconfig.get_config_var('SOABI'),'machine':platform.machine(),
    'include_dir':sysconfig.get_path('include'),'Python_h_exists':__import__('pathlib').Path(sysconfig.get_path('include'),'Python.h').is_file(),
    'all_distributions':records,'module_specs':modules,'wheel_tags':tags}))
'''


def run_command(command, timeout=30):
    env = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
           "HF_HUB_DISABLE_TELEMETRY": "1", "PIP_NO_INDEX": "1", "USE_TF": "0", "USE_FLAX": "0",
           "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True", "DIFFUSERS_OFFLINE": "1",
           "HF_DATASETS_OFFLINE": "1", "PYTHONDONTWRITEBYTECODE": "1", "PYTHONUTF8": "1",
           "PIP_DISABLE_PIP_VERSION_CHECK": "1", "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"}
    for key in ('PYTHONHOME', 'PYTHONPATH', 'PYTHONUSERBASE'):
        env.pop(key, None)
    try:
        result = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace",
                                timeout=timeout, env=env)
        return {"status": "ok" if result.returncode == 0 else "failed", "returncode": result.returncode,
                "stdout": result.stdout, "stderr": result.stderr}
    except subprocess.TimeoutExpired as exc:
        def decode(value):
            return value.decode("utf-8", errors="replace") if isinstance(value, bytes) else value or ""
        return {"status": "timeout", "timeout_seconds": timeout,
                "stdout": decode(exc.stdout), "stderr": decode(exc.stderr)}
    except OSError as exc:
        return {"status": "unavailable", "detail": str(exc)}


def read_text(path):
    try:
        return Path(path).read_text(encoding="utf-8").strip()
    except OSError:
        return None


def hardware():
    meminfo = {}
    for line in (read_text("/proc/meminfo") or "").splitlines():
        key, value = line.split(":", 1)
        meminfo[key] = int(value.strip().split()[0]) * 1024
    limit = read_text("/sys/fs/cgroup/memory.max") or read_text("/sys/fs/cgroup/memory/memory.limit_in_bytes")
    current = read_text("/sys/fs/cgroup/memory.current") or read_text("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    limit = int(limit) if limit and limit.isdigit() else None
    current = int(current) if current and current.isdigit() else None
    totals = [value for value in (meminfo.get("MemTotal"), limit) if value is not None]
    total = min(totals) if totals else None
    available = meminfo.get("MemAvailable")
    if limit is not None and current is not None:
        available = min(available if available is not None else limit, max(0, limit-current))
    quota = read_text("/sys/fs/cgroup/cpu.max")
    try:
        affinity = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None
    except OSError:
        affinity = None
    cpu_model = next((line.split(":",1)[1].strip() for line in (read_text("/proc/cpuinfo") or "").splitlines()
                      if line.startswith("model name")), platform.processor())
    disks = {}
    for name in ("/", "/kaggle/input", "/kaggle/working", "/tmp"):
        if Path(name).is_dir():
            disk = shutil.disk_usage(name)
            disks[name] = {"total_GiB": round(disk.total/1024**3,2), "free_GiB": round(disk.free/1024**3,2)}
    return {"os": platform.platform(), "machine": platform.machine(), "libc": platform.libc_ver(),
            "python": sys.version, "python_executable": sys.executable, "python_ABI": sysconfig.get_config_var("SOABI"),
            "cpu_model": cpu_model, "cpu_logical": os.cpu_count(), "cpu_affinity": affinity, "cpu_quota_raw": quota,
            "RAM_effective_GiB": round(total/1024**3,2) if total is not None else None,
            "RAM_available_GiB": round(available/1024**3,2) if available is not None else None, "disks": disks}


def distributions():
    records = []
    seen = set()
    for dist in metadata.distributions():
        name = dist.metadata.get('Name', '')
        key = name.lower().replace('_', '-').replace('.', '-')
        records.append({"name": name, "version": dist.version, 'active': key not in seen,
                        'location': str(dist.locate_file('')),
                        "requires_python": dist.metadata.get("Requires-Python"), "requires": dist.requires or []})
        seen.add(key)
    return sorted(records, key=lambda item: item["name"].lower())


def offline_inputs(root=Path("/kaggle/input"), max_dirs=150, max_files=80):
    found, visited = [], 0
    if not root.is_dir():
        return {"exists": False, "files": [], "truncated": False}
    for parent, dirs, files in os.walk(root):
        visited += 1
        if len(Path(parent).relative_to(root).parts) >= 6:
            dirs[:] = []
        for name in sorted(files):
            if name.endswith((".whl", ".tar.gz", ".tar.xz", ".zip", ".safetensors", ".pth")) or name in {"config.json", "inference.yml", "bundle_manifest.json"}:
                path = Path(parent) / name
                try:
                    size = path.stat().st_size
                except OSError:
                    continue
                found.append({"path": str(path), "bytes": size})
                if len(found) >= max_files:
                    return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
        if visited >= max_dirs:
            return {"exists": True, "files": found, "truncated": True, "visited_dirs": visited}
    return {"exists": True, "files": found, "truncated": False, "visited_dirs": visited}


def diagnostic_options(options=None):
    if set(options or {}) - DIAGNOSTIC_DEFAULTS.keys():
        raise ValueError('Unknown diagnostic options')
    result = {**DIAGNOSTIC_DEFAULTS, **(options or {})}
    for key, default in DIAGNOSTIC_DEFAULTS.items():
        if isinstance(default, bool):
            if type(result[key]) is not bool:
                raise ValueError(f'{key} must be bool')
        elif type(result[key]) is not int or result[key] < 1:
            raise ValueError(f'{key} must be a positive integer')
    return result


def diagnostic_json(path):
    path = Path(path)
    if path.stat().st_size > 16*1024*1024:
        raise ValueError('Diagnostic metadata exceeds 16 MiB; not read')
    return json.loads(path.read_text(encoding='utf8'))


def diagnostic_payload(output):
    for line in reversed(output.splitlines()):
        try:
            value = json.loads(line)
            if isinstance(value, dict):
                return value
        except ValueError:
            continue
    return None


def discover_asset_roots(input_root, configured, *, max_dirs=500):
    """Bounded discovery prunes weights/objects/source trees; explicit paths take priority."""
    markers = {'models': 'models_manifest.json', 'packages': 'bundle_manifest.json',
               'prepared_images': 'dataset.json'}
    candidates = {key: [] for key in markers}
    root, visited, truncated = Path(input_root), 0, False
    if root.is_dir():
        for parent, dirs, files in os.walk(root):
            visited += 1
            for kind, marker in markers.items():
                if marker in files:
                    candidates[kind].append(str(Path(parent).resolve()))
            dirs[:] = sorted(d for d in dirs if d not in {'sources', 'wheels', 'objects', 'snapshots', '__pycache__', '.cache'})
            if len(Path(parent).relative_to(root).parts) >= 6:
                dirs[:] = []
            if visited >= max_dirs:
                truncated = True
                break
    found = {}
    for kind, marker in markers.items():
        explicit = configured.get(kind)
        if explicit:
            path = Path(explicit).resolve()
            if path.is_file():
                path = path.parent.parent if path.parent.name == 'snapshots' else path.parent
            for child in (path/kind, path/'kaggle_upload'/kind):
                if not (path/marker).is_file() and (child/marker).is_file():
                    path = child
                    break
            found[kind] = {'path': str(path), 'selection': 'explicit', 'exists': path.is_dir()}
        elif len(candidates[kind]) == 1:
            found[kind] = {'path': candidates[kind][0], 'selection': 'discovered', 'exists': True}
        else:
            found[kind] = {'path': None, 'selection': 'ambiguous' if candidates[kind] else 'not_found',
                           'candidates': candidates[kind], 'exists': False}
    found['discovery'] = {'visited_dirs': visited, 'truncated': truncated}
    found['pdfs'] = {'path': str(configured['pdfs']) if configured.get('pdfs') else None,
                     'selection': 'explicit' if configured.get('pdfs') else 'not_configured'}
    return found


def diagnostic_relative(root, relative):
    value = str(relative)
    parts = PurePosixPath(value)
    if ':' in value or '\\' in value or parts.is_absolute() or '..' in parts.parts:
        raise ValueError('Invalid portable asset path')
    root = Path(root).resolve()
    path = root.joinpath(*parts.parts)
    if not path.resolve().is_relative_to(root):
        raise ValueError('Asset reference escapes its root')
    return path


def inspect_asset_records(root, records, *, limit=12000, verify_hashes=False):
    """Default checks file presence and size only, without opening model weights."""
    missing, mismatches, checked, bytes_declared, hashes_verified = [], [], 0, 0, 0
    for record in records[:limit]:
        relative = record.get('path', '')
        try:
            path = diagnostic_relative(root, relative)
            if not path.is_file():
                missing.append(relative)
                continue
            actual = path.stat().st_size
            if record.get('bytes') is not None and actual != record['bytes']:
                mismatches.append({'path': relative, 'kind': 'size', 'actual': actual, 'expected': record['bytes']})
            elif verify_hashes and record.get('sha256'):
                digest = hashlib.sha256()
                with path.open('rb') as stream:
                    for block in iter(lambda: stream.read(8*1024*1024), b''):
                        digest.update(block)
                if digest.hexdigest() != record['sha256']:
                    mismatches.append({'path': relative, 'kind': 'sha256'})
                else:
                    hashes_verified += 1
            checked += 1
            bytes_declared += record.get('bytes', actual)
        except (OSError, ValueError) as exc:
            mismatches.append({'path': relative, 'kind': 'unreadable_or_invalid', 'detail': str(exc)})
    return {'status': 'issues_found' if missing or mismatches else 'analysis_incomplete' if len(records) > limit
            else 'files_present_sizes_match', 'declared_files': len(records), 'checked_files': checked,
            'missing': missing, 'mismatches': mismatches, 'declared_bytes_checked': bytes_declared,
            'truncated': len(records) > limit, 'hashes_requested': verify_hashes,
            'hashes_verified_files': hashes_verified,
            'model_inference_verified': False}


def inspect_offline_assets(roots, settings):
    reports = {}
    def inspect(name, root, manifest, required=()):
        result = {'manifest': str(manifest), 'required_missing': [], 'state': 'manifest_missing'}
        try:
            data = diagnostic_json(manifest)
            result.update(inspect_asset_records(root, data.get('files', []), limit=settings['max_asset_records'],
                                                verify_hashes=settings['verify_asset_hashes']))
            result['state'] = result['status']
            result['required_missing'] = [r for r in required if not diagnostic_relative(root, r).is_file()]
            indexed = {r.get('path') for r in data.get('files', [])}
            result['required_not_in_inventory'] = [r for r in required if r not in indexed]
            if result['required_missing'] or result['required_not_in_inventory']:
                result['state'] = 'missing_required_asset'
            if name == 'tadisr':
                shards = []
                for prefix, index in (('transformer', 'diffusion_pytorch_model.safetensors.index.json'),
                                      ('text_encoder', 'model.safetensors.index.json')):
                    index_path = Path(root)/'CogView4'/prefix/index
                    if index_path.is_file():
                        body = diagnostic_json(index_path)
                        for part in set(body['weight_map'].values()):
                            if (not diagnostic_relative(Path(root)/'CogView4'/prefix, part).is_file()
                                    or f'CogView4/{prefix}/{part}' not in indexed):
                                shards.append(prefix+'/'+part)
                result['missing_indexed_shards'] = shards
                if shards:
                    result['state'] = 'missing_required_asset'
                result['prompt_embeddings_present'] = (Path(root)/'CogView4/saved_prompt_tokens_nocfg.pt').is_file()
                result['prompt_note'] = 'Notebook 1 can derive the fixed prompt offline from verified base assets'
        except (OSError, ValueError, KeyError, TypeError) as exc:
            result['detail'] = str(exc)
            if Path(manifest).is_file():
                result['state'] = 'invalid_manifest'
        reports[name] = result
    models = roots.get('models', {}).get('path')
    if models:
        models = Path(models)
        inspect('ocr_models', models, models/'models_manifest.json', ['tonggu/layout/finetune_infer.pth'])
        requirements = {
            'realesrgan': ['RealESRGAN_x4plus.pth'],
            'difftsr': ['DiffTSR.ckpt', 'transocr.pth'],
            'unihir': ['model.safetensors', 'ae.safetensors', 'llm_config.json', 'vit_config.json',
                      'tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt'] +
                     ['font/'+n for n in ('KaiXinSongA.ttf','KaiXinSongB.ttf','TH-Tshyn-P0.ttf','TH-Tshyn-P1.ttf','TH-Tshyn-P16.ttf','TH-Tshyn-P2.ttf')],
            'tadisr': ['tadisr_cogview4_realce_17500.pkl'] + ['CogView4/'+n for n in (
                'model_index.json','vae/config.json','vae/diffusion_pytorch_model.safetensors',
                'transformer/config.json','transformer/diffusion_pytorch_model.safetensors.index.json',
                'text_encoder/config.json','text_encoder/model.safetensors.index.json',
                'tokenizer/tokenizer.json','tokenizer/tokenizer_config.json','tokenizer/special_tokens_map.json',
                'scheduler/scheduler_config.json')],
        }
        for name, required in requirements.items():
            root = models/'image_restoration'/name
            inspect(name, root, root/'assets.json', required)
    packages = roots.get('packages', {}).get('path')
    if packages:
        packages = Path(packages)
        inspect('ocr_packages', packages, packages/'bundle_manifest.json',
                ['paddle_manifest.json','tonggu_manifest.json','system_assets_manifest.json'])
        restoration = packages/'image_restoration'
        try:
            plan = diagnostic_json(restoration/'manifests/tadisr_runtime.json')
            reports['tadisr_packages'] = inspect_asset_records(restoration, plan.get('wheels', []),
                limit=settings['max_asset_records'], verify_hashes=settings['verify_asset_hashes'])
        except (OSError, ValueError, TypeError) as exc:
            reports['tadisr_packages'] = {'status': 'manifest_unavailable', 'detail': str(exc)}
        for name in ('difftsr', 'unihir'):
            try:
                reports[name+'_runtime_manifest'] = diagnostic_json(restoration/f'manifests/{name}_runtime.json')
            except (OSError, ValueError) as exc:
                reports[name+'_runtime_manifest'] = {'state': 'manifest_unavailable', 'detail': str(exc)}
    prepared = roots.get('prepared_images', {}).get('path')
    if prepared:
        try:
            data = diagnostic_json(Path(prepared)/'dataset.json')
            reports['prepared_images'] = {'format': data.get('format'), 'version': data.get('version'),
                'snapshot_id': data.get('snapshot_id'), 'pages': len(data.get('pages', [])),
                'restoration': data.get('restoration'), 'state': 'descriptor_read_only', 'pixels_verified': False}
        except (OSError, ValueError, TypeError) as exc:
            reports['prepared_images'] = {'state': 'descriptor_unavailable', 'detail': str(exc)}
    return reports


def runtime_candidates(configured, *, include_existing=True):
    result = {str(name): {'python': str(path), 'source': 'explicit'} for name, path in (configured or {}).items() if path}
    if include_existing:
        for path in (Path('/tmp/vtm_kaggle_runtime/runtime_config.json'), Path('/kaggle/working/runtime_config.json')):
            try:
                data = diagnostic_json(path)
                for name in ('paddle', 'tonggu'):
                    if data.get(name+'_python') and name not in result:
                        result[name] = {'python': data[name+'_python'], 'source': str(path)}
            except (OSError, ValueError, TypeError):
                pass
        path = Path('/kaggle/working/.image_runtime/preflight.json')
        try:
            for name, item in diagnostic_json(path).items():
                if isinstance(item, dict) and item.get('python') and name not in result:
                    result[name] = {'python': item['python'], 'source': str(path)}
        except (OSError, ValueError, TypeError):
            pass
    return result


def runtime_probe_names(name):
    common = ['torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'pdf_renderer', 'torch_build']
    if name in {'paddle', 'ppocr'}:
        return ['numpy_cv2', 'pdf_renderer', 'paddle_cuda', 'paddleocr_apis']
    if name == 'tonggu':
        return common + ['torchvision_ops', 'mmcv_compiled_ops']
    if name == 'difftsr':
        return common + ['torchvision_ops', 'difftsr_dependencies']
    if name in {'unihir', 'unihir_restoration'}:
        return common + ['unihir_dependencies', 'flash_attention', 'torch_attention']
    if name in {'tadisr', 'tadisr_x4'}:
        return common + ['diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies', 'torch_restoration_ops']
    if name in {'realesrgan', 'realesrgan_x4'}:
        return common + ['torch_restoration_ops']
    return common


def inspect_staged_wheels(roots, runtime_inventories):
    packages = roots.get('packages', {}).get('path')
    if not packages:
        return {'state': 'packages_not_configured', 'wheels': []}
    try:
        from packaging.utils import parse_wheel_filename
        result = []
        for folder in (Path(packages)/'wheels', Path(packages)/'image_restoration/wheels'):
            for path in sorted(folder.glob('*.whl'))[:250]:
                try:
                    _, _, _, tags = parse_wheel_filename(path.name)
                except ValueError as exc:
                    result.append({'file': str(path), 'state': 'invalid_wheel_filename', 'detail': str(exc)})
                    continue
                wheel_tags = {str(tag) for tag in tags}
                result.append({'file': str(path), 'wheel_tags': sorted(wheel_tags),
                    'compatible_with_observed_interpreters': {name: bool(wheel_tags & set(inv['wheel_tags']))
                        if inv.get('wheel_tags') else None for name, inv in runtime_inventories.items()}})
        return {'state': 'tag_comparison_only', 'wheels': result,
                'note': 'A cp310 wheel failing kernel cp313 tags can still belong to a separate cp310 runtime; tags do not prove CUDA support'}
    except ImportError:
        return {'state': 'packaging_module_unavailable', 'wheels': []}


def readiness_summary(report):
    """Per-test observations only: never promote these diagnostics to model readiness."""
    groups = {'notebook_1_classical': ['numpy_cv2', 'pdf_renderer'],
        'notebook_1_realesrgan': ['torch_cuda', 'torch_numpy_bridge', 'numpy_cv2', 'torch_restoration_ops'],
        'notebook_1_tadisr': ['torch_cuda', 'diffusers_cogview4', 'transformers_glm', 'peft_api', 'restoration_dependencies'],
        'notebook_1_difftsr': ['torch_cuda', 'difftsr_dependencies'],
        'notebook_1_unihir': ['torch_cuda', 'unihir_dependencies', 'flash_attention'],
        'notebook_3_hunyuan': ['torch_cuda', 'torch_numpy_bridge', 'transformers_hunyuan'],
        'notebook_3_paddle_in_kernel': ['paddle_cuda', 'paddleocr_apis'],
        'notebook_3_tonggu_in_kernel': ['torch_cuda', 'mmcv_compiled_ops']}
    result = {}
    for name, probes in groups.items():
        states = {p: report['probes'].get(p, {}).get('status', 'not_run') for p in probes}
        result[name] = {'kernel_probe_states': states, 'model_inference_verified': False,
            'note': 'Kernel imports are separate from staged assets and private runtimes. Passing tiny ops does not verify full model inference.'}
    return result


def collect_environment(*, run_probes=True, input_root=Path('/kaggle/input'), configured_paths=None,
                        runtime_pythons=None, options=None, checkpoint_path=None):
    settings = diagnostic_options(options)
    started = time.monotonic()
    inventory = distributions()
    versions = {}
    for item in inventory:
        name = item['name'].lower().replace('_', '-')
        if name not in versions or item.get('active') is True:
            versions[name] = item['version']
    roots = discover_asset_roots(input_root, configured_paths or {}, max_dirs=settings['max_scan_dirs'])
    report = {"schema": 1, "diagnostics_version": 2, "collected_at_UTC": datetime.now(timezone.utc).isoformat(),
              "hardware": hardware(), "packages": {name: versions.get(name) for name in PACKAGES},
              "all_distributions": inventory, "probes": {}, "tools": {}, "offline_inputs": offline_inputs(Path(input_root)),
              "prior_vtm_runtime_exists": Path("/tmp/vtm_kaggle_runtime/runtime_config.json").is_file(),
              "asset_roots": roots, "options": settings, "assets": inspect_offline_assets(roots, settings),
              "runtimes": {}, "complete": False, "status": 'collecting', "execution_limits": [],
              "model_inference_performed": False,
              "configuration": {'input_root': str(input_root), 'configured_paths': {k:str(v) for k,v in (configured_paths or {}).items()},
                                'runtime_pythons': {k:str(v) for k,v in (runtime_pythons or {}).items()}},
              "build_inventory": {'Python_include': sysconfig.get_path('include'),
                  'Python_h_exists': Path(sysconfig.get_path('include'), 'Python.h').is_file(),
                  'sysconfig': {k:sysconfig.get_config_var(k) for k in ('CC','CXX','LDSHARED','LIBDIR','EXT_SUFFIX','SOABI')},
                  'selected_environment': {k:os.environ.get(k) for k in ('CUDA_HOME','CUDA_PATH','CC','CXX','CUDACXX','MAX_JOBS','CUDA_VISIBLE_DEVICES')}},
              "note": "Diagnostics only. Metadata/imports/tiny ops; no installation, downloads, model loading or inference. Assets use size checks unless hashes explicitly enabled."}
    def checkpoint():
        report['elapsed_seconds'] = round(time.monotonic()-started, 3)
        if checkpoint_path:
            path = Path(checkpoint_path)
            if path.resolve().is_relative_to(Path('/kaggle/input').resolve()):
                raise ValueError('Diagnostic report must not be written to Kaggle Input')
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary = path.with_name(path.name+'.tmp')
            temporary.write_text(json.dumps(report, ensure_ascii=False, indent=2)+'\n', encoding='utf8')
            temporary.replace(path)
    def bounded(command, timeout=30):
        remaining = settings['time_budget_seconds']-(time.monotonic()-started)
        if remaining < 1:
            report['complete'] = False
            if 'time_budget_seconds' not in report['execution_limits']:
                report['execution_limits'].append('time_budget_seconds')
            return {'status': 'not_run_budget', 'detail': 'Diagnostic time budget exhausted; partial report retained'}
        return run_command(command, min(timeout, settings['probe_timeout_cap'], max(1, int(remaining))))
    checkpoint()
    if not run_probes:
        report['readiness'] = readiness_summary(report)
        report.update(complete=True, status='inventory_only')
        checkpoint()
        return report
    commands = {"gpu": ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
                "nvcc": ["nvcc", "--version"], "g++": ["g++", "--version"],
                "cmake": ["cmake", "--version"], "ninja": ["ninja", "--version"],
                "gcc": ["gcc", "--version"], "ldd": ["ldd", "--version"]}
    for name, command in commands.items():
        report["tools"][name] = bounded(command)
        checkpoint()
    report["pip_check"] = bounded([sys.executable, "-m", "pip", "check"], timeout=60)
    report['kernel_runtime_inventory'] = bounded([sys.executable, '-c', PROBE_GUARD+RUNTIME_INVENTORY], timeout=60)
    kernel_inventory = diagnostic_payload(report['kernel_runtime_inventory'].get('stdout', '')) or {}
    observed = {'kernel': kernel_inventory}
    report['kernel_runtime_inventory']['data'] = kernel_inventory
    checkpoint()
    for name, (module, timeout, code) in PROBES.items():
        print(f"Checking {name} (timeout {timeout}s)...", flush=True)
        try:
            present = importlib.util.find_spec(module) is not None
        except (ImportError, ValueError):
            present = False
        result = bounded([sys.executable, "-c", PROBE_GUARD+code], timeout) if present else {"status": "not_installed"}
        result['data'] = diagnostic_payload(result.get('stdout', ''))
        report["probes"][name] = result
        print(f"  {result['status']}", flush=True)
        checkpoint()
    candidates = runtime_candidates(runtime_pythons, include_existing=settings['check_existing_runtimes'])
    seen_python = {str(Path(sys.executable).resolve()): 'kernel'}
    checked = 0
    for name, candidate in candidates.items():
        python = str(Path(candidate['python']).expanduser().resolve())
        state = {**candidate, 'python': python, 'model_inference_verified': False, 'probes': {}}
        report['runtimes'][name] = state
        if python in seen_python:
            state.update(status='shared_interpreter', shares_with=seen_python[python])
            checkpoint()
            continue
        if checked >= settings['max_runtime_checks']:
            state['status'] = 'not_run_runtime_limit'
            report['complete'] = False
            if 'max_runtime_checks' not in report['execution_limits']:
                report['execution_limits'].append('max_runtime_checks')
            continue
        if not Path(python).is_file():
            state['status'] = 'interpreter_missing'
            continue
        checked += 1
        seen_python[python] = name
        print(f'Checking separate runtime {name}: {python}', flush=True)
        state['inventory'] = bounded([python, '-c', PROBE_GUARD+RUNTIME_INVENTORY], timeout=60)
        detail = diagnostic_payload(state['inventory'].get('stdout', ''))
        if state['inventory']['status'] != 'ok' or detail is None:
            state['status'] = 'inventory_unavailable'
            checkpoint()
            continue
        state['inventory']['data'] = detail
        observed[name] = detail
        state['pip_check'] = bounded([python, '-m', 'pip', 'check'], timeout=60)
        for probe_name in runtime_probe_names(name):
            module, timeout, code = PROBES[probe_name]
            present = detail.get('module_specs', {}).get(module, {}).get('available', False)
            print(f'  {name}/{probe_name}', flush=True)
            probe = bounded([python, '-c', PROBE_GUARD+code], timeout) if present else {'status':'not_installed'}
            probe['data'] = diagnostic_payload(probe.get('stdout', ''))
            state['probes'][probe_name] = probe
            checkpoint()
        state['status'] = 'probed; model_inference_unverified'
        checkpoint()
    report['staged_wheel_tags'] = inspect_staged_wheels(roots, observed)
    report['readiness'] = readiness_summary(report)
    report['limitations'] = ['Missing kernel packages may be supplied by a separate staged runtime.',
        'Wheel ABI tags and tiny GPU ops do not verify full model execution or restoration fidelity.',
        'DiffTSR/UniHIR model code and weights are not executed by this notebook.',
        'No compatible runtime is automatically installed, compiled or downloaded.']
    report['complete'] = not report['execution_limits']
    report['status'] = 'completed_with_probe_results' if report['complete'] else 'partial_limits'
    checkpoint()
    return report


def save_report(report, output_dir):
    output = Path(output_dir)
    if output.resolve().is_relative_to(Path('/kaggle/input').resolve()):
        raise ValueError('Output must be outside Kaggle Input')
    output.mkdir(parents=True, exist_ok=True)
    json_path = output / "kaggle_environment_report.json"
    text_path = output / "kaggle_environment_report.txt"
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    lines = [json.dumps(report["hardware"], ensure_ascii=False, indent=2), "\nPackages:"]
    lines.extend(f"{name:24s} {version or 'not installed'}" for name, version in report["packages"].items())
    for group in ("tools", "probes"):
        for name, result in report[group].items():
            lines.append(f"\n[{name}]\n" + json.dumps(result, ensure_ascii=False, indent=2))
    lines.append("\n[pip_check]\n" + json.dumps(report.get("pip_check"), ensure_ascii=False, indent=2))
    for name in ('asset_roots', 'assets', 'runtimes', 'build_inventory', 'readiness', 'limitations'):
        if name in report:
            lines.append(f'\n[{name}]\n'+json.dumps(report[name], ensure_ascii=False, indent=2))
    text_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return json_path, text_path




## 1. Thu thập cấu hình và kiểm tra các API/tensor nhỏ

Không cần sửa ô mã hỗ trợ phía trên. `failed`, `timeout`, `not_installed` được giữ
riêng. Nếu có nhiều Python, kết quả từng interpreter nằm ở `runtimes`; không lấy
package của Python 3.13 làm bằng chứng cho Python 3.10. Wheel tags chỉ kiểm tra
tương thích ABI/platform, chưa xác nhận native CUDA hoặc inference.


In [ ]:
report = collect_environment(
    run_probes=RUN_PROBES, input_root=INPUT_ROOT,
    configured_paths={'models':MODELS_DIR, 'packages':PACKAGES_DIR,
                      'pdfs':DATA_DIR, 'prepared_images':PREPARED_IMAGES},
    runtime_pythons=RUNTIME_PYTHONS, options=CHECK_OPTIONS,
    checkpoint_path=OUTPUT_DIR/'kaggle_environment_report.json')
print(json.dumps(report["hardware"], ensure_ascii=False, indent=2))
print()
print("GPU:", report["tools"].get("gpu", {}).get("stdout", "unavailable"))
print("Torch/CUDA:", report["probes"].get("torch_cuda", {}).get("stdout", "not available"))
print()
print("Các package liên quan:")
for name, version in report["packages"].items():
    print(f"{name:24s} {version or 'not installed'}")
print()
print("Pip dependency check:", report.get("pip_check", {}).get("status"))
print("Runtime VTM từ phiên trước:", report["prior_vtm_runtime_exists"])
print('Trạng thái báo cáo:', report['status'])
print('Input:', json.dumps(report['asset_roots'], ensure_ascii=False, indent=2))
print('Asset:')
for name, details in report['assets'].items():
    print(name, details.get('state', details.get('status')), details.get('required_missing', []))
print('Runtime riêng:', {name:value['status'] for name,value in report['runtimes'].items()})


## 2. Gửi lại báo cáo để hoàn thiện bộ offline

Gửi lại **`kaggle_environment_report.json`** trong Output sau khi chạy xong.
File TXT chứa log dễ đọc. Nếu bị ngắt, JSON đã lưu một phần vẫn hữu ích; trường
`status`/`complete` cho biết các phép kiểm tra đã kết thúc hay chưa.

Báo cáo giữ toàn bộ distributions/dependencies, ABI, vị trí module thật, log lỗi,
GPU architecture, compiler/header và file asset còn thiếu. Từ dữ liệu này mới
chọn package/runtime cần bổ sung cho DiffTSR/UniHIR mà không hạ kernel đang dùng.
Không cần upload lại models/packages chỉ để lấy báo cáo. Nếu Input vừa cập nhật
UniHIR, gắn đúng phiên bản Dataset mới để kiểm tra checkpoint và inventory tương ứng.


In [ ]:
json_path, text_path = save_report(report, OUTPUT_DIR)
print("Báo cáo JSON:", json_path)
print("Báo cáo TXT:", text_path)
try:
    from IPython.display import display, FileLink
    display(FileLink(str(json_path)))
    display(FileLink(str(text_path)))
except ImportError:
    pass
